# Mapping multiple Olist source codes to Texas ZIPs

This notebook maps each Olist `source_code` block to one Texas ZIP using cumulative weighted capacity ranges. The final step joins the mapping back to the customer-level Olist table and writes a new CSV without editing the original source files.

In [17]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 80

PROJECT_ROOT = Path.cwd().parent
OLIST_CUSTOMERS_CSV = PROJECT_ROOT / "Olist E-Commerce Dataset" / "olist_customers_dataset.csv"
OLIST_CUSTOMERS_WITH_SOURCE_CSV = PROJECT_ROOT / "Olist E-Commerce Dataset" / "olist_customers_with_source_code.csv"
SOURCE_CODE_SUMMARY_CSV = PROJECT_ROOT / "Establish_source_codes_and_target_weights" / "source_code_summary.csv"
TEXAS_TARGET_WEIGHTS_CSV = PROJECT_ROOT / "Establish_source_codes_and_target_weights" / "texas_target_weights.csv"

OUTPUT_MAPPING_CSV = PROJECT_ROOT / "Synthetic_Customers_and_Orders" / "postal_customers.csv"
OUTPUT_CUSTOMERS_WITH_ZIP_CSV = PROJECT_ROOT / "Synthetic_Customers_and_Orders" / "olist_customers_with_assigned_texas_zip.csv"


In [18]:
# Main three requested dataframes.
olist_customers_df = pd.read_csv(
    OLIST_CUSTOMERS_CSV,
    dtype={"customer_zip_code_prefix": "string"},
)
source_code_summary_df = pd.read_csv(
    SOURCE_CODE_SUMMARY_CSV,
    dtype={"source_code": "string"},
)
texas_weight_df = pd.read_csv(
    TEXAS_TARGET_WEIGHTS_CSV,
    dtype={"zip": "string", "county_geoid": "string"},
)

olist_customers_df["customer_zip_code_prefix"] = olist_customers_df["customer_zip_code_prefix"].str.zfill(5)
texas_weight_df["zip"] = texas_weight_df["zip"].str.zfill(5)

# The raw Olist customer CSV has no source_code column. Use the enriched customer file for the final join.
if "source_code" in olist_customers_df.columns:
    olist_customers_for_mapping_df = olist_customers_df.copy()
elif OLIST_CUSTOMERS_WITH_SOURCE_CSV.exists():
    olist_customers_for_mapping_df = pd.read_csv(
        OLIST_CUSTOMERS_WITH_SOURCE_CSV,
        dtype={
            "customer_zip_code_prefix": "string",
            "zip_group_4": "string",
            "source_code": "string",
            "zip_group_3": "string",
            "zip_group_2": "string",
        },
    )
else:
    raise FileNotFoundError(
        "The raw customer CSV does not contain source_code, and "
        f"{OLIST_CUSTOMERS_WITH_SOURCE_CSV.name} was not found."
    )

print(f"olist_customers_df: {olist_customers_df.shape}")
print(f"source_code_summary_df: {source_code_summary_df.shape}")
print(f"texas_weight_df: {texas_weight_df.shape}")
print(f"olist_customers_for_mapping_df: {olist_customers_for_mapping_df.shape}")


olist_customers_df: (99441, 5)
source_code_summary_df: (5699, 4)
texas_weight_df: (1915, 13)
olist_customers_for_mapping_df: (99441, 12)


In [19]:
required_source_cols = {"source_code"}
required_texas_cols = {"zip", "target_zip_capacity"}

missing_source_cols = required_source_cols - set(source_code_summary_df.columns)
missing_texas_cols = required_texas_cols - set(texas_weight_df.columns)

if missing_source_cols:
    raise ValueError(f"source_code_summary_df is missing columns: {sorted(missing_source_cols)}")
if missing_texas_cols:
    raise ValueError(f"texas_weight_df is missing columns: {sorted(missing_texas_cols)}")

if "source_code_count" not in source_code_summary_df.columns:
    if "count" not in source_code_summary_df.columns:
        raise ValueError("source_code_summary_df must contain either 'source_code_count' or 'count'.")
    source_code_summary_df["source_code_count"] = source_code_summary_df["count"]

source_code_summary_df["source_code_count"] = pd.to_numeric(
    source_code_summary_df["source_code_count"],
    errors="raise",
)
texas_weight_df["target_zip_capacity"] = pd.to_numeric(
    texas_weight_df["target_zip_capacity"],
    errors="raise",
)

# Largest source blocks and largest Texas ZIP capacities are packed first.
source_code_summary_df = source_code_summary_df.sort_values(
    ["source_code_count", "source_code"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)
texas_weight_df = texas_weight_df.sort_values(
    ["target_zip_capacity", "zip"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

source_code_summary_df["source_start"] = source_code_summary_df["source_code_count"].cumsum().shift(fill_value=0)
source_code_summary_df["source_end"] = source_code_summary_df["source_code_count"].cumsum()
source_code_summary_df["source_midpoint"] = (
    source_code_summary_df["source_start"] + source_code_summary_df["source_end"]
) / 2

texas_weight_df["target_start"] = texas_weight_df["target_zip_capacity"].cumsum().shift(fill_value=0)
texas_weight_df["target_end"] = texas_weight_df["target_zip_capacity"].cumsum()

source_total = source_code_summary_df["source_code_count"].sum()
target_total = texas_weight_df["target_zip_capacity"].sum()

print(f"Total Olist source customers: {source_total:,.0f}")
print(f"Total Texas target capacity: {target_total:,.2f}")
print(f"Difference: {target_total - source_total:,.6f}")

display(source_code_summary_df.head())
display(texas_weight_df.head())


Total Olist source customers: 99,441
Total Texas target capacity: 99,441.00
Difference: 0.000000


,source_code,count,proportion,percentage,source_code_count,source_start,source_end,source_midpoint
0,SP-1321,366,0.003681,0.368057,366,0,366,183.0
1,RJ-2279,334,0.003359,0.335878,334,366,700,533.0
2,SP-1308,257,0.002584,0.258445,257,700,957,828.5
3,ES-2910,253,0.002544,0.254422,253,957,1210,1083.5
4,MG-3840,248,0.002494,0.249394,248,1210,1458,1334.0


,zip,latitude,longitude,city,state,county_geoid,county_name,population,density,centroid_source,target_weight,target_zip_proportion,target_zip_capacity,target_start,target_end
0,77494,29.74566,-95.82302,KATY,TX,48157,Fort Bend,140157,1428.4,uszips,361152.812135,0.005861,582.780469,0.000000,582.780469
1,77449,29.83674,-95.73547,KATY,TX,48201,Harris,130028,1856.7,uszips,357755.290123,0.005805,577.297999,582.780469,1160.078469
2,78660,30.44361,-97.59558,PFLUGERVILLE,TX,48453,Travis,124834,1080.5,uszips,299987.270778,0.004868,484.079638,1160.078469,1644.158106
3,77084,29.82698,-95.66120,HOUSTON,TX,48201,Harris,110217,1341.0,uszips,279556.456965,0.004536,451.111102,1644.158106,2095.269209
4,79936,31.77373,-106.29631,EL PASO,TX,48141,El Paso,102991,1535.4,uszips,270220.570472,0.004385,436.046088,2095.269209,2531.315296


In [20]:
target_ends = texas_weight_df["target_end"].to_numpy()
source_midpoints = source_code_summary_df["source_midpoint"].to_numpy()

# Find the first Texas ZIP cumulative endpoint that contains each source_code midpoint.
assigned_target_positions = np.searchsorted(target_ends, source_midpoints, side="left")
assigned_target_positions = np.clip(assigned_target_positions, 0, len(texas_weight_df) - 1)
assigned_texas_rows = texas_weight_df.iloc[assigned_target_positions].reset_index(drop=True)

mapping_table_df = pd.DataFrame(
    {
        "source_code": source_code_summary_df["source_code"].to_numpy(),
        "source_customer_count": source_code_summary_df["source_code_count"].to_numpy(),
        "source_start": source_code_summary_df["source_start"].to_numpy(),
        "source_end": source_code_summary_df["source_end"].to_numpy(),
        "source_midpoint": source_code_summary_df["source_midpoint"].to_numpy(),
        "assigned_texas_zip": assigned_texas_rows["zip"].to_numpy(),
        "assigned_texas_city": assigned_texas_rows.get("city", pd.Series([pd.NA] * len(assigned_texas_rows))).to_numpy(),
        "assigned_texas_county": assigned_texas_rows.get("county_name", pd.Series([pd.NA] * len(assigned_texas_rows))).to_numpy(),
        "target_zip_capacity": assigned_texas_rows["target_zip_capacity"].to_numpy(),
        "target_start": assigned_texas_rows["target_start"].to_numpy(),
        "target_end": assigned_texas_rows["target_end"].to_numpy(),
    }
)

zip_assignment_summary_df = (
    mapping_table_df.groupby("assigned_texas_zip", as_index=False)
    .agg(
        assigned_source_code_count=("source_code", "count"),
        assigned_customer_count=("source_customer_count", "sum"),
    )
    .merge(
        texas_weight_df[["zip", "target_zip_capacity", "target_start", "target_end"]],
        left_on="assigned_texas_zip",
        right_on="zip",
        how="right",
    )
    .drop(columns="zip")
)
zip_assignment_summary_df["assigned_source_code_count"] = zip_assignment_summary_df["assigned_source_code_count"].fillna(0).astype(int)
zip_assignment_summary_df["assigned_customer_count"] = zip_assignment_summary_df["assigned_customer_count"].fillna(0).astype(int)
zip_assignment_summary_df["capacity_minus_assigned_customers"] = (
    zip_assignment_summary_df["target_zip_capacity"] - zip_assignment_summary_df["assigned_customer_count"]
)

# Sub-1 capacity ZIPs can only receive customers when a whole source_code midpoint lands in their range.
tiny_zip_assignment_df = zip_assignment_summary_df[zip_assignment_summary_df["target_zip_capacity"] < 1].copy()

print(f"Mapped source codes: {len(mapping_table_df):,}")
print(f"Texas ZIPs receiving at least one source_code: {(zip_assignment_summary_df['assigned_source_code_count'] > 0).sum():,}")
print(f"Sub-1 capacity ZIPs receiving at least one source_code: {(tiny_zip_assignment_df['assigned_source_code_count'] > 0).sum():,}")

display(mapping_table_df.head(10))
display(zip_assignment_summary_df.head(10))


Mapped source codes: 5,699
Texas ZIPs receiving at least one source_code: 1,630
Sub-1 capacity ZIPs receiving at least one source_code: 150


,source_code,source_customer_count,source_start,source_end,source_midpoint,assigned_texas_zip,assigned_texas_city,assigned_texas_county,target_zip_capacity,target_start,target_end
0,SP-1321,366,0,366,183.0,77494,KATY,Fort Bend,582.780469,0.000000,582.780469
1,RJ-2279,334,366,700,533.0,77494,KATY,Fort Bend,582.780469,0.000000,582.780469
2,SP-1308,257,700,957,828.5,77449,KATY,Harris,577.297999,582.780469,1160.078469
3,ES-2910,253,957,1210,1083.5,77449,KATY,Harris,577.297999,582.780469,1160.078469
4,MG-3840,248,1210,1458,1334.0,78660,PFLUGERVILLE,Travis,484.079638,1160.078469,1644.158106
5,SP-1224,242,1458,1700,1579.0,78660,PFLUGERVILLE,Travis,484.079638,1160.078469,1644.158106
6,SP-1170,240,1700,1940,1820.0,77084,HOUSTON,Harris,451.111102,1644.158106,2095.269209
7,SP-1305,239,1940,2179,2059.5,77084,HOUSTON,Harris,451.111102,1644.158106,2095.269209
8,SP-1223,236,2179,2415,2297.0,79936,EL PASO,El Paso,436.046088,2095.269209,2531.315296
9,SP-1304,215,2415,2630,2522.5,79936,EL PASO,El Paso,436.046088,2095.269209,2531.315296


,assigned_texas_zip,assigned_source_code_count,assigned_customer_count,target_zip_capacity,target_start,target_end,capacity_minus_assigned_customers
0,77494,2,700,582.780469,0.000000,582.780469,-117.219531
1,77449,2,510,577.297999,582.780469,1160.078469,67.297999
2,78660,2,490,484.079638,1160.078469,1644.158106,-5.920362
3,77084,2,479,451.111102,1644.158106,2095.269209,-27.888898
4,79936,2,451,436.046088,2095.269209,2531.315296,-14.953912
5,77433,2,410,414.319503,2531.315296,2945.634799,4.319503
6,75052,2,373,403.912590,2945.634799,3349.547389,30.912590
7,77573,2,363,396.270086,3349.547389,3745.817475,33.270086
8,77407,2,352,383.890656,3745.817475,4129.708131,31.890656
9,77036,2,336,381.354720,4129.708131,4511.062851,45.354720


In [21]:
if "source_code" not in olist_customers_for_mapping_df.columns:
    raise ValueError("olist_customers_for_mapping_df must contain source_code before joining the mapping table.")

missing_mapping_source_codes = sorted(
    set(olist_customers_for_mapping_df["source_code"].dropna().unique())
    - set(mapping_table_df["source_code"].dropna().unique())
)
if missing_mapping_source_codes:
    raise ValueError(
        f"{len(missing_mapping_source_codes):,} customer source_codes were not found in mapping_table_df. "
        f"Examples: {missing_mapping_source_codes[:10]}"
    )

olist_customers_with_assigned_zip_df = olist_customers_for_mapping_df.merge(
    mapping_table_df[["source_code", "assigned_texas_zip"]],
    on="source_code",
    how="left",
    validate="many_to_one",
)

missing_assigned_zip_count = olist_customers_with_assigned_zip_df["assigned_texas_zip"].isna().sum()
if missing_assigned_zip_count:
    raise ValueError(f"{missing_assigned_zip_count:,} customer rows did not receive an assigned_texas_zip.")

# mapping_table_df.to_csv(OUTPUT_MAPPING_CSV, index=False)
# olist_customers_with_assigned_zip_df.to_csv(OUTPUT_CUSTOMERS_WITH_ZIP_CSV, index=False)

print(f"Wrote mapping table: {OUTPUT_MAPPING_CSV}")
print(f"Wrote customer table with assigned Texas ZIPs: {OUTPUT_CUSTOMERS_WITH_ZIP_CSV}")
print(f"Customer rows with assigned_texas_zip: {len(olist_customers_with_assigned_zip_df):,}")

display(olist_customers_with_assigned_zip_df.head())


Wrote mapping table: z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic_Customers_and_Orders\postal_customers.csv
Wrote customer table with assigned Texas ZIPs: z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic_Customers_and_Orders\olist_customers_with_assigned_texas_zip.csv
Customer rows with assigned_texas_zip: 99,441


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,zip_group_4,source_code,zip_group_3,zip_group_2,source_code_count,source_code_proportion,source_code_percentage,assigned_texas_zip
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1440,SP-1440,144,14,155,0.001559,0.155871,75035
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,0979,SP-0979,097,09,40,0.000402,0.040225,77079
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,0115,SP-0115,011,01,46,0.000463,0.046259,76210
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,0877,SP-0877,087,08,67,0.000674,0.067377,75025
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,1305,SP-1305,130,13,239,0.002403,0.240344,77084


In [22]:
quality_check_df = pd.DataFrame(
    {
        "check": [
            "source rows mapped",
            "source customer total",
            "customer rows assigned",
            "unique Texas ZIPs assigned",
            "Texas ZIP target total",
            "sub-1 capacity ZIPs",
            "sub-1 capacity ZIPs assigned",
        ],
        "value": [
            len(mapping_table_df),
            mapping_table_df["source_customer_count"].sum(),
            len(olist_customers_with_assigned_zip_df),
            mapping_table_df["assigned_texas_zip"].nunique(),
            texas_weight_df["target_zip_capacity"].sum(),
            len(tiny_zip_assignment_df),
            (tiny_zip_assignment_df["assigned_source_code_count"] > 0).sum(),
        ],
    }
)

display(quality_check_df)
display(tiny_zip_assignment_df[tiny_zip_assignment_df["assigned_source_code_count"] > 0].head(20))


,check,value
0,source rows mapped,5699.0
1,source customer total,99441.0
2,customer rows assigned,99441.0
3,unique Texas ZIPs assigned,1630.0
4,Texas ZIP target total,99441.0
5,sub-1 capacity ZIPs,435.0
6,sub-1 capacity ZIPs assigned,150.0


,assigned_texas_zip,assigned_source_code_count,assigned_customer_count,target_zip_capacity,target_start,target_end,capacity_minus_assigned_customers
1480,78402,1,1,0.999367,99291.392330,99292.391697,-0.000633
1481,76351,1,1,0.997369,99292.391697,99293.389066,-0.002631
1482,79543,1,1,0.994560,99293.389066,99294.383626,-0.005440
1483,75478,1,1,0.993328,99294.383626,99295.376954,-0.006672
1484,75560,1,1,0.987238,99295.376954,99296.364192,-0.012762
1485,75566,1,1,0.985013,99296.364192,99297.349205,-0.014987
1486,79788,1,1,0.976501,99297.349205,99298.325706,-0.023499
1487,78884,1,1,0.975364,99298.325706,99299.301070,-0.024636
1488,78870,1,1,0.960289,99299.301070,99300.261359,-0.039711
1489,79778,1,1,0.956688,99300.261359,99301.218047,-0.043312


In [23]:
postal_customers = olist_customers_with_assigned_zip_df.drop(columns=[
    "customer_city",
    "customer_zip_code_prefix",
    "customer_state", 
    "zip_group_4", 
    "source_code", 
    "zip_group_3", 
    "zip_group_2", 
    "source_code_count", 
    "source_code_proportion", 
    "source_code_percentage", 
    ])
postal_customers.head(10)

,customer_id,customer_unique_id,assigned_texas_zip
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,75035
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,77079
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,76210
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,75025
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,77084
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,75061
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,79912
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,78666
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,76541
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,78665


# Mapping Texas ZIP codes to Texas Geographical Information
**Now that we have assigned every Olist customer a Texas ZIP, we need a centralized CSV that contains not only the Texas ZIP, but also...**

Names (Can be randomly generated)
    - First name
    - Middle Initial
    - Last name

Locational Data
    - Street Address (Randomly generated or location based parameters? Still needs to be decided)
    - City
    - State (All TX)
    - ZIP Code

Personal Info
    - Phone number (Prefix must be ZIP related? Can just have placeholders for now)
    - Email (random too long as not used for user_roles / login)
    - User_id (should we create an auto update procedure to give every customer a user?)
    - Preferred_facility_id (needs to be calculated fromp procedure)
    - Birth Date
    - Marital status
    - Gender
    - Annual Income
    - Total Children
    - Education Level
    - Occupation
    - Home Owner


In [ ]:
# postal_customers.to_csv(OUTPUT_MAPPING_CSV, index=False)